# Live RAG Pipeline

Compare generated answers with ground truth

Configuration:
- Chunking: Semantic-Level, 192 tokens
- Retrieval: Dense MMR, k=8, λ=0.5
- Document Order: Reverse (ascending relevance)
- Generator: Llama 3.1 8B

## Setup

In [1]:
!pip install -q python-dotenv datasets tiktoken langchain-core langchain-text-splitters langchain-huggingface langchain-chroma langchain-openai nltk


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import json
import numpy as np
import pandas as pd
import tiktoken
import tempfile
from dotenv import load_dotenv
from datasets import load_dataset
from nltk.tokenize import sent_tokenize
import nltk
from typing import List

from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

nltk.download('punkt_tab', quiet=True)
load_dotenv()
openrouter_token = os.environ.get('OPENROUTER_TOKEN')

print("✓ Imports complete")

/Users/saikrishna/Desktop/Codespace/IIIT_AI_ML_course/Capstone_Project/reliablerag/.venv-1/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Imports complete


## Load Dataset

In [ ]:
print("Loading dataset...")
DATASET_NAME = "delucionqa"  # feeds both the loader below and vector store naming
ds = load_dataset("galileo-ai/ragbench", DATASET_NAME, split="test")[:50]
df = pd.DataFrame(ds)

# Flatten documents
all_docs = []
for _, row in df.iterrows():
    for doc_pos, doc_text in enumerate(row["documents"]):
        all_docs.append({
            "doc_id": f"{row['id']}_d{doc_pos}",
            "row_id": str(row["id"]),
            "text": doc_text.strip(),
        })

docs_df = pd.DataFrame(all_docs)
print(f"✓ Loaded {len(df)} queries, {len(docs_df)} documents")

## Setup Models

In [ ]:
import sys, os


def _add_ragbench_lib_to_path():
    for candidate in (os.getcwd(), os.path.join(os.getcwd(), "delucion_dataset")):
        if os.path.isdir(os.path.join(candidate, "ragbench_lib")) and candidate not in sys.path:
            sys.path.insert(0, candidate)
            return


_add_ragbench_lib_to_path()

from ragbench_lib.models import get_embedding_model, get_generation_llm
from ragbench_lib.generation_prompt import RAG_GENERATION_PROMPT
from ragbench_lib.vector_store import get_persist_dir, vector_store_names

embedding_model = get_embedding_model(openrouter_token)

llm = get_generation_llm(openrouter_token)

prompt = RAG_GENERATION_PROMPT

print("✓ Models ready")


## Semantic Chunking

In [ ]:
from ragbench_lib.chunking import count_tokens, create_semantic_chunks

print("Creating chunks...")
documents = create_semantic_chunks(docs_df, target_tokens=192)
print(f"✓ Created {len(documents)} chunks")


## Setup Retriever

In [ ]:
class LiveRetriever:
    def __init__(self, documents, embedding_model, k=8, dataset_name="delucionqa"):
        self.k = k
        prefix, collection_name = vector_store_names(dataset_name, "live")
        persist_dir = get_persist_dir(prefix)
        self.vector_store = Chroma.from_documents(
            documents=documents,
            embedding=embedding_model,
            collection_name=collection_name,
            persist_directory=persist_dir,
        )
    
    def retrieve(self, query):
        """Retrieve with reverse ordering"""
        results = self.vector_store.similarity_search_with_score(query, k=self.k)
        docs_with_scores = [(doc, 1 - score) for doc, score in results]
        sorted_docs = sorted(docs_with_scores, key=lambda x: x[1], reverse=False)
        return [doc for doc, score in sorted_docs]

print("Setting up retriever...")
retriever = LiveRetriever(documents, embedding_model, k=8, dataset_name=DATASET_NAME)
print("✓ Retriever ready")

## Live Pipeline

In [7]:
print("\n" + "="*100)
print("LIVE RAG PIPELINE - ANSWER COMPARISON")
print("="*100 + "\n")

for idx, row in df.iterrows():
    question = row["question"]
    ground_truth = row["response"]
    
    try:
        # Retrieve
        docs = retriever.retrieve(question)
        context = "\n\n".join([d.page_content for d in docs])
        
        # Generate
        generated = (prompt | llm | StrOutputParser()).invoke({
            "context": context,
            "question": question
        })
        
        # Display
        print(f"\n{'─'*100}")
        print(f"Query #{idx+1}/{len(df)}")
        print(f"{'─'*100}")
        
        print(f"\n❓ QUESTION:")
        print(f"   {question}")
        
        print(f"\n🤖 GENERATED ANSWER:")
        print(f"   {generated}")
        
        print(f"\n✓ GROUND TRUTH ANSWER:")
        print(f"   {ground_truth}")
        
        print()
        
    except Exception as e:
        print(f"\n❌ Query #{idx+1} - Error: {str(e)[:100]}")

print("\n" + "="*100)
print("PIPELINE COMPLETE")
print("="*100)


LIVE RAG PIPELINE - ANSWER COMPARISON


────────────────────────────────────────────────────────────────────────────────────────────────────
Query #1/50
────────────────────────────────────────────────────────────────────────────────────────────────────

❓ QUESTION:
   What if I fail to latch the tailgate properly?

🤖 GENERATED ANSWER:
   If you fail to latch the tailgate properly, it could result in damage to the vehicle or cargo.

✓ GROUND TRUTH ANSWER:
   If you fail to securely latch the tailgate properly, it could result in damage to the vehicle or cargo.


────────────────────────────────────────────────────────────────────────────────────────────────────
Query #2/50
────────────────────────────────────────────────────────────────────────────────────────────────────

❓ QUESTION:
   What kind of safety features are implemented in this car?

🤖 GENERATED ANSWER:
   Based on the provided context, the following safety features are implemented in this car:

1. Adaptive Cruise Control 